<!-- notebook-header -->
# Probabilidade I: Fundamentos para ML

**Modulo:** 00 - Matematica  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Eventos, probabilidade condicional, Bayes, variaveis aleatorias e distribuicoes fundamentais.


# Probabilidade I: Fundamentos para ML

**Tempo estimado:** 10-12 horas
**Pre-requisitos:** 0.1 (Fundamentos), 0.5 (Integrais)
**Proximo modulo:** 0.7 (Probabilidade Avancada)

---

## Indice

1. Introducao: Probabilidade como linguagem da incerteza
2. Espaco amostral, eventos e axiomas de Kolmogorov
3. Regras de probabilidade (Uniao, Complemento)
4. Probabilidade condicional
5. Regra de Bayes
6. Independencia de eventos e Naive Bayes
7. Variaveis aleatorias discretas (PMF, CDF)
8. Distribuicao Bernoulli
9. Distribuicao Binomial
10. Distribuicao Poisson
11. Variaveis aleatorias continuas (PDF, CDF)
12. Distribuicao Normal (Gaussiana)
13. Distribuicao Exponencial
14. Esperanca e Variancia
15. Lei dos Grandes Numeros
16. Exercicios finais
17. Erros Comuns
18. Resumo e Conexoes

## Pre-requisitos e Fio Narrativo

| Topico | Notebook | Por que precisa |
|--------|----------|-----------------|
| Conjuntos e logica | 0.1 | Eventos sao conjuntos; axiomas usam uniao/intersecao |
| Integrais | 0.5 | PDF continua: P(a < X < b) = integral de f(x) |
| Series | 0.5 | PMF discreta: soma infinita deve convergir para 1 |

**Fio narrativo deste notebook:**

Probabilidade e a linguagem matematica da incerteza. Em ML, quase tudo e probabilistico: os dados sao amostras de distribuicoes desconhecidas, os modelos estimam probabilidades, e as predicoes vem com incerteza.

Neste notebook, construiremos a teoria em camadas:

1. **Base axiomatica** (secoes 2-3): as regras do jogo
2. **Condicionamento e Bayes** (secoes 4-6): como aprender com evidencia
3. **Distribuicoes** (secoes 7-13): os "personagens" da probabilidade
4. **Resumos numericos** (secoes 14-15): como descrever distribuicoes

Ao final, voce tera as ferramentas para entender modelos probabilisticos como Naive Bayes, regressao logistica, e redes generativas.

## Por que Probabilidade em ML?

**Analogia:** Probabilidade e como uma balanca de evidencias. Cada dado que observamos pesa a favor ou contra uma hipotese.

Onde probabilidade aparece em ML:

| Conceito | Exemplo em ML | Secao |
|----------|---------------|-------|
| P(classe dado features) | Classificador Naive Bayes | 5-6 |
| Distribuicao dos dados | Normalidade dos erros em regressao | 12 |
| Atualizacao com evidencia | Inferencia Bayesiana | 5 |
| Incerteza nas predicoes | Intervalos de confianca | 14-15 |
| Geracao de dados sinteticos | VAEs, Diffusion Models | 11-13 |

**Conexao com 0.5:** La usamos integrais para calcular areas. Aqui, area sob uma curva de densidade = probabilidade. A integral e a ferramenta, probabilidade e a interpretacao.

## 1. Introducao: Probabilidade como Linguagem da Incerteza

Probabilidade responde a pergunta: **"Quao provavel e X?"**

Existem duas interpretacoes principais:

- **Frequentista:** probabilidade = frequencia relativa em repeticoes infinitas. "Se lancar um dado infinitas vezes, 1/6 das vezes sai 3."
- **Bayesiana:** probabilidade = grau de crenca. "Acredito com 70% de confianca que vai chover amanha."

Em ML, ambas as visoes sao uteis:
- Frequentista: justifica treinamento com muitos dados (Lei dos Grandes Numeros)
- Bayesiana: permite incorporar conhecimento previo (priors) e atualizar com dados

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# from scipy import stats
import seaborn as sns

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 2. Espaco Amostral, Eventos e Axiomas de Kolmogorov

### Intuicao: As Regras do Jogo

Imagine que voce vai organizar todas as apostas possiveis num jogo. Precisa de 3 regras para que ninguem trapaceie:

1. **Nao-negatividade:** nenhuma aposta pode ter probabilidade negativa (P(A) >= 0)
2. **Normalizacao:** a soma de todas as apostas possiveis e 100% (P(omega) = 1)
3. **Aditividade:** se dois resultados nunca ocorrem juntos, suas probabilidades se somam

Estes 3 axiomas, propostos por Kolmogorov em 1933, sao suficientes para derivar TODA a teoria de probabilidade.

**Definicoes formais:**

| Conceito | Definicao | Exemplo (dado) |
|----------|-----------|-----------------|
| Espaco amostral (omega) | Todos os resultados possiveis | {1, 2, 3, 4, 5, 6} |
| Evento | Subconjunto de omega | "sair par" = {2, 4, 6} |
| Probabilidade P | Funcao omega -> [0,1] satisfazendo os 3 axiomas | P("par") = 3/6 = 0.5 |

**Por que em ML:** Quando definimos um classificador, o espaco amostral sao todas as classes possiveis, e P(classe|dados) deve satisfazer os axiomas (somar 1, ser nao-negativa).

In [ ]:
# Simulação: lançar um dado 100.000 vezes
num_simulations = 100_000
rolls = np.random.randint(1, 7, size=num_simulations)

# Contar frequências
results, counts = np.unique(rolls, return_counts=True)
empirical_probs = counts / num_simulations
theoretical_probs = np.ones(6) / 6

fig, ax = plt.subplots()
width = 0.35
ax.bar(results - width/2, empirical_probs, width, label='Empírica', alpha=0.8)
ax.bar(results + width/2, theoretical_probs, width, label='Teórica (1/6)', alpha=0.8)
ax.set_xlabel('Resultado do dado')
ax.set_ylabel('Probabilidade')
ax.set_title('Axioma de Kolmogorov: Convergência da frequência para probabilidade')
ax.set_xticks(results)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Soma das probabilidades empíricas: {empirical_probs.sum():.4f}')
print(f'Satisfaz axioma 2: P(Ω) = 1')
print(f'Todas P(i) ≥ 0: {np.all(empirical_probs >= 0)}')
print(f'Satisfaz axioma 1 e 2 ✓')

**O que observar:** As barras empiricas e teoricas devem se alinhar quase perfeitamente. A frequencia de cada face converge para 1/6 ≈ 0.1667.

**O que concluir:** Os axiomas de Kolmogorov nao sao arbitrarios -- emergem naturalmente dos dados. Com 100.000 lancamentos, a realidade se adequa aos 3 axiomas. A soma das probabilidades empiricas e praticamente 1, e todas sao nao-negativas.

**Conexao com secao 15:** Esta convergencia e formalizada pela Lei dos Grandes Numeros.

## 3. Regras de Probabilidade (Uniao e Complemento)

### Intuicao: Evitando Dupla-Contagem

Imagine que voce quer contar quantas cartas num baralho sao vermelhas OU ases. Se simplesmente somar "vermelhas + ases", vai contar os ases vermelhos duas vezes. A regra da uniao corrige isso:

**Regra da Uniao:**
$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

**Regra do Complemento:** Se algo tem 60% de chance de ocorrer, tem 40% de chance de NAO ocorrer:
$$P(A^c) = 1 - P(A)$$

**Por que em ML:** Quando calculamos P(erro), muitas vezes e mais facil calcular P(acerto) e subtrair de 1. A regra do complemento e usada constantemente em funcoes de perda.

In [ ]:
# Simulação de baralho: P(carta vermelha OU ás)
# Baralho: 52 cartas, 26 vermelhas (13 corações + 13 ouros), 4 ases (1 em cada naipe)
# P(vermelho) = 26/52 = 0.5
# P(ás) = 4/52
# P(vermelho E ás) = 2/52 (ás de ouros e ás de corações)

p_red = 26 / 52
p_ace = 4 / 52
p_red_and_ace = 2 / 52

# Regra da União
p_red_or_ace_formula = p_red + p_ace - p_red_and_ace

# Verificar por simulação
cartas = np.arange(1, 53)  # 1-26: vermelhas, 27-52: pretas; 1,14,27,40: ases
is_red = (cartas <= 26)
is_ace = np.isin(cartas, [1, 14, 27, 40])
is_red_or_ace = is_red | is_ace

p_red_or_ace_empirical = is_red_or_ace.sum() / 52

print(f'P(Vermelho) = {p_red:.4f}')
print(f'P(Ás) = {p_ace:.4f}')
print(f'P(Vermelho E Ás) = {p_red_and_ace:.4f}')
print(f'\nP(Vermelho OU Ás) pela fórmula: {p_red_or_ace_formula:.4f}')
print(f'P(Vermelho OU Ás) por contagem: {p_red_or_ace_empirical:.4f}')
print(f'Regra da União verificada: {np.isclose(p_red_or_ace_formula, p_red_or_ace_empirical)}')

# Regra do Complemento
print(f'\nP(Não vermelho) = {1 - p_red:.4f}')
print(f'P(Preto) = {(~is_red).sum() / 52:.4f}')
print(f'Regra do Complemento verificada ✓')

**O que observar:** Os valores calculados pela formula coincidem exatamente com a contagem direta. A regra funciona tanto analiticamente quanto empiricamente.

**O que concluir:** A regra da uniao garante consistencia: sem ela, poderiamos "inventar" probabilidade ao contar eventos que se sobrepoe. O complemento e um atalho poderoso -- em vez de somar muitos eventos, calculamos 1 menos o oposto.

**Conexao com secao 4:** A probabilidade condicional refina essas regras ao considerar informacao adicional.

## 4. Probabilidade Condicional

### Intuicao: De "Geral" para "Pessoal"

Sem informacao: qual a chance de um email ser spam? Cerca de 10%. Esta e a probabilidade *a priori*.

Agora voce ve que o email contem "CLICK AQUI AGORA!!!". A chance de ser spam sobe drasticamente. Esta e a probabilidade *condicional*:

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

Leia como: "probabilidade de A, dado que B ocorreu."

**Por que em ML:** Todo classificador aprende P(Y|X) -- a probabilidade da classe Y dado as features X. Treinamento e essencialmente estimar essas probabilidades condicionais a partir dos dados.

| Antes (Prior) | Evidencia | Depois (Posterior) |
|---------------|-----------|-------------------|
| P(spam) = 10% | "click aqui" | P(spam dado "click aqui") ≈ 64% |
| P(gato) = 5% | imagem com orelhas pontudas | P(gato dado orelhas) >> 5% |

In [ ]:
# Dataset: 10.000 emails
num_emails = 10_000
spam_prob = 0.1
legit_prob = 0.9

# Gerar labels
is_spam = np.random.binomial(1, spam_prob, num_emails)

# Gerar "click aqui" condicionado ao label
has_click = np.zeros(num_emails, dtype=bool)
has_click[is_spam == 1] = np.random.binomial(1, 0.8, is_spam.sum())  # 80% de spam
has_click[is_spam == 0] = np.random.binomial(1, 0.05, (is_spam == 0).sum())  # 5% de legítimos

# Cálculo: P(Spam | "click aqui")
has_click_and_spam = (has_click & (is_spam == 1)).sum()
has_click_total = has_click.sum()

p_spam_given_click = has_click_and_spam / has_click_total

print(f'P(Spam) = {is_spam.mean():.4f}')
print(f'P("click aqui") = {has_click.mean():.4f}')
print(f'P("click aqui" | Spam) = {has_click[is_spam == 1].mean():.4f}')
print(f'P("click aqui" | Legítimo) = {has_click[is_spam == 0].mean():.4f}')
print(f'\nP(Spam | "click aqui") = {p_spam_given_click:.4f}')
print(f'\nInterpretação: Se um email contém "click aqui",\nhá {p_spam_given_click*100:.1f}% de chance de ser spam!')

# Visualizar
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição incondicional
ax1.bar(['Legítimo', 'Spam'], [legit_prob, spam_prob], color=['green', 'red'], alpha=0.7)
ax1.set_ylabel('Probabilidade')
ax1.set_title('P(Email)')
ax1.set_ylim([0, 1])

# Distribuição condicional
spam_with_click = has_click[is_spam == 1].mean()
legit_with_click = has_click[is_spam == 0].mean()
ax2.bar(['Legítimo', 'Spam'], [legit_with_click, spam_with_click], color=['green', 'red'], alpha=0.7)
ax2.set_ylabel('Probabilidade')
ax2.set_title('P("click aqui" | Email)')
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.show()

**O que observar:** O grafico esquerdo mostra as probabilidades incondicionais (P(spam) ≈ 10%). O grafico direito mostra P("click aqui" | tipo): spam tem ~80% e legitimo tem ~5%. O valor calculado P(spam | "click aqui") e muito maior que o prior.

**O que concluir:** Observar uma pista ("click aqui") multiplica drasticamente a chance de spam. O mecanismo e simples: "click aqui" e muito mais frequente em spam do que em emails legitimos, entao quando vemos essa pista, a probabilidade de spam dispara.

**Conexao com secao 5:** Bayes formaliza essa "inversao" -- dado que vemos a pista, qual e a causa?

## 5. Regra de Bayes

### Intuicao: Invertendo a Pergunta

Voce quer saber: "Estou doente?" (causa → efeito desconhecido)

O que o teste mede: "Se estiver doente, o teste da positivo?" (causa conhecida → efeito)

Bayes inverte a direcao:

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

| Componente | Nome | Significado |
|------------|------|-------------|
| P(A dado B) | **Posterior** | O que queremos saber (atualizado) |
| P(B dado A) | **Likelihood** | Quao provavel e a evidencia se A for verdade |
| P(A) | **Prior** | Crenca antes de ver a evidencia |
| P(B) | **Evidence** | Probabilidade total da evidencia |

**Por que em ML:** Bayes e o fundamento de:
- Classificadores Naive Bayes (secao 6)
- Inferencia Bayesiana (0.7)
- Redes Bayesianas
- Atualizacao online de modelos

**Armadilha classica:** Um teste com 95% de sensibilidade pode ter posterior baixissimo se a doenca for rara. O prior "ancora" a posterior.

In [ ]:
# Exemplo: Teste COVID
# Suponha que você fez um teste COVID que tem:
# - Sensibilidade (True Positive Rate) = 95%: P(+ | infectado) = 0.95
# - Especificidade (True Negative Rate) = 99%: P(- | não infectado) = 0.99
# - Prevalência na população = 1%: P(infectado) = 0.01

p_infected = 0.01  # prior
p_positive_given_infected = 0.95  # sensitivity (likelihood)
p_positive_given_not_infected = 1 - 0.99  # 1 - specificity
p_not_infected = 1 - p_infected

# Lei da probabilidade total: P(+) = P(+|I)P(I) + P(+|¬I)P(¬I)
p_positive = (p_positive_given_infected * p_infected + 
              p_positive_given_not_infected * p_not_infected)

# Regra de Bayes
p_infected_given_positive = (p_positive_given_infected * p_infected) / p_positive

print(f'Prior P(Infectado) = {p_infected:.4f} (1% da população)')
print(f'Likelihood P(+ | Infectado) = {p_positive_given_infected:.4f} (Sensibilidade)')
print(f'P(+ | Não infectado) = {p_positive_given_not_infected:.4f}')
print(f'Evidence P(+) = {p_positive:.4f}')
print(f'\nPosterior P(Infectado | +) = {p_infected_given_positive:.4f}')
print(f'\nInterpretação importante:')
print(f'Apesar do teste ser 95% sensível, se você testar positivo,\nhá apenas {p_infected_given_positive*100:.1f}% de chance de estar realmente infectado!')
print(f'Por quê? Porque a doença é rara (1%), então falsos positivos\ncompõem a maioria dos resultados positivos.')

# Visualizar com scipy
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Prior
axes[0].bar(['Não infectado', 'Infectado'], [p_not_infected, p_infected], color=['blue', 'red'], alpha=0.7)
axes[0].set_ylabel('Probabilidade')
axes[0].set_title('Prior: P(Infectado)')
axes[0].set_ylim([0, 1])

# Likelihood
axes[1].bar(['Infectado', 'Não infectado'], [p_positive_given_infected, p_positive_given_not_infected], color=['red', 'blue'], alpha=0.7)
axes[1].set_ylabel('P(+ | Estado)')
axes[1].set_title('Likelihood: P(+ | Estado)')
axes[1].set_ylim([0, 1])

# Posterior
axes[2].bar(['Não infectado', 'Infectado'], [1-p_infected_given_positive, p_infected_given_positive], color=['blue', 'red'], alpha=0.7)
axes[2].set_ylabel('Probabilidade')
axes[2].set_title('Posterior: P(Infectado | +) - ATUALIZOU!')
axes[2].set_ylim([0, 1])

plt.tight_layout()
plt.show()

**O que observar:** As tres figuras mostram a "jornada Bayesiana": Prior (1% infectado) → Likelihood (teste 95% sensivel) → Posterior (~49%). O prior baixo "puxa" a posterior para baixo, mesmo com teste bom.

**O que concluir:** Este resultado e contra-intuitivo e extremamente importante:
- Teste com 95% de sensibilidade + doenca com 1% de prevalencia = apenas ~49% de chance de estar doente dado teste positivo
- A maioria dos positivos sao falsos positivos (vindos dos 99% saudaveis)
- Implicacao clinica: sempre confirmar com segundo teste

**Conexao com secao 6:** Naive Bayes aplica essa mesma logica a classificacao de textos, multiplicando likelihoods de cada feature.

## 6. Independencia de Eventos e Naive Bayes

### Intuicao: O Truque que Simplifica Tudo

Dois eventos A e B sao **independentes** se saber que A ocorreu nao muda a probabilidade de B:

$$P(A \cap B) = P(A) \cdot P(B) \quad \Leftrightarrow \quad P(A|B) = P(A)$$

**O truque do Naive Bayes:** Para classificar um documento com 100 palavras, precisariamos estimar P(palavra1, palavra2, ..., palavra100 | classe). Isso requer uma tabela com 2^100 entradas -- impossivel.

Assumindo independencia condicional:
$$P(w_1, w_2, ..., w_{100} | classe) = \prod_{i=1}^{100} P(w_i | classe)$$

Agora precisamos apenas de 100 probabilidades por classe -- totalmente praticavel.

**Por que funciona apesar de ser "ingenue"?** Porque o que importa para classificacao e a *ordem relativa* das probabilidades, nao seus valores exatos. Mesmo com a suposicao errada, o ranking geralmente esta correto.

In [ ]:
# Exemplo: Features independentes no Naive Bayes
# Dataset: 1000 documentos, classificar como SPAM ou HAM (legítimo)
# Feature 1: contém "dinheiro" - P(Feature1 | SPAM) = 0.6, P(Feature1 | HAM) = 0.1
# Feature 2: contém "click aqui" - P(Feature2 | SPAM) = 0.7, P(Feature2 | HAM) = 0.05

num_docs = 10_000
p_feature1_spam = 0.6
p_feature1_ham = 0.1
p_feature2_spam = 0.7
p_feature2_ham = 0.05

# Gerar dados
is_spam = np.random.binomial(1, 0.3, num_docs)
feature1 = np.zeros(num_docs, dtype=bool)
feature2 = np.zeros(num_docs, dtype=bool)

for i in range(num_docs):
    if is_spam[i]:
        feature1[i] = np.random.random() < p_feature1_spam
        feature2[i] = np.random.random() < p_feature2_spam
    else:
        feature1[i] = np.random.random() < p_feature1_ham
        feature2[i] = np.random.random() < p_feature2_ham

# Verificar independência: P(F1 & F2 | SPAM) ≈ P(F1 | SPAM) * P(F2 | SPAM)?
spam_mask = is_spam == 1
p_f1_and_f2_given_spam = (feature1[spam_mask] & feature2[spam_mask]).sum() / spam_mask.sum()
p_f1_given_spam = feature1[spam_mask].mean()
p_f2_given_spam = feature2[spam_mask].mean()
p_f1_f2_product = p_f1_given_spam * p_f2_given_spam

print('Verificando independência em documentos SPAM:')
print(f'P(Feature1 & Feature2 | SPAM) = {p_f1_and_f2_given_spam:.4f}')
print(f'P(Feature1 | SPAM) * P(Feature2 | SPAM) = {p_f1_f2_product:.4f}')
print(f'Diferença: {abs(p_f1_and_f2_given_spam - p_f1_f2_product):.4f}')
print(f'Independência verificada: {np.isclose(p_f1_and_f2_given_spam, p_f1_f2_product, atol=0.05)}')
print(f'\nNaive Bayes assume essa independência para simplificar cálculos!'),

# Visualização
fig, ax = plt.subplots()
labels = ['Feature1 & Feature2\n(empírica)', 'Feature1 * Feature2\n(independência)']
values = [p_f1_and_f2_given_spam, p_f1_f2_product]
colors = ['steelblue', 'orange']
ax.bar(labels, values, color=colors, alpha=0.7)
ax.set_ylabel('Probabilidade | SPAM')
ax.set_title('Verificação de Independência em Naive Bayes')
ax.set_ylim([0, max(values) * 1.2])
for i, v in enumerate(values):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center')
plt.tight_layout()
plt.show()

**O que observar:** As duas barras (empirica vs produto das marginais) devem estar muito proximas. A diferenca e pequena, confirmando que as features geradas sao de fato independentes.

**O que concluir:** Quando geramos features independentes, P(F1 e F2 | spam) ≈ P(F1 | spam) * P(F2 | spam). Na pratica, features reais raramente sao perfeitamente independentes, mas Naive Bayes e surpreendentemente robusto a essa violacao.

**Conexao com 0.7:** No notebook de Probabilidade Avancada, veremos como relaxar essa suposicao com distribuicoes conjuntas e multivariadas.

## 7. Variaveis Aleatorias Discretas

### Intuicao: Discreto = Contavel

Uma variavel aleatoria discreta X toma valores num conjunto contavel: X = 0, 1, 2, 3, ...

Duas formas de descreve-la:

| Funcao | Pergunta que responde | Formula |
|--------|----------------------|---------|
| **PMF** (massa) | Qual a chance de exatamente k? | P(X = k) |
| **CDF** (acumulada) | Qual a chance de no maximo k? | F(k) = P(X <= k) |

**Relacao:** CDF e a soma acumulada da PMF. Se voce sabe a PMF, pode calcular a CDF (e vice-versa).

**Por que em ML:** Quando treinamos um classificador com k classes, a saida e uma PMF sobre as classes (vetor softmax). A classe predita e o argmax dessa PMF.

In [ ]:
# Exemplo: Número de emails spam em 20 emails aleatórios
# X ~ Binomial(n=20, p=0.3)

n_trials = 20
p_spam = 0.3

# Gerar amostras
samples = np.random.binomial(n_trials, p_spam, 10_000)

# PMF
values, counts = np.unique(samples, return_counts=True)
pmf = counts / counts.sum()

# CDF
cdf = np.cumsum(pmf)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(values, pmf, alpha=0.7, color='steelblue')
ax1.set_xlabel('Número de emails spam')
ax1.set_ylabel('P(X = k)')
ax1.set_title('PMF: Função Massa de Probabilidade')
ax1.grid(axis='y', alpha=0.3)

ax2.step(values, cdf, where='mid', color='red', linewidth=2, label='CDF empírica')
ax2.set_xlabel('Número de emails spam')
ax2.set_ylabel('P(X ≤ x)')
ax2.set_title('CDF: Função Distribuição Acumulada')
ax2.grid(alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print(f'P(X = 6) = {pmf[values == 6][0]:.4f}')
print(f'P(X ≤ 6) = {cdf[values == 6][0]:.4f}')
print(f'P(X > 6) = {1 - cdf[values == 6][0]:.4f}')

**O que observar:** A PMF tem formato de sino com pico proximo a n*p = 6. A CDF cresce em degraus de 0 ate 1, sendo a soma acumulada da PMF.

**O que concluir:** PMF e CDF sao duas lentes para a mesma distribuicao. PMF e util para probabilidades pontuais ("exatamente 6 spams"). CDF e util para probabilidades acumuladas ("no maximo 6 spams" ou "mais de 6 spams" = 1 - CDF).

**Conexao com secao 11:** Para variaveis continuas, PMF vira PDF e a relacao se mantem, mas com integrais em vez de somas.

## 8. Distribuicao Bernoulli

### Intuicao: O Atomo da Probabilidade

Bernoulli e a distribuicao mais simples possivel: um unico experimento com dois resultados.

$$P(X = 1) = p, \quad P(X = 0) = 1 - p$$

| Propriedade | Valor |
|-------------|-------|
| E[X] | p |
| Var[X] | p(1-p) |
| Variancia maxima | quando p = 0.5 (maxima incerteza) |

**Por que em ML:** Cada predicao binaria (spam/nao-spam, clique/nao-clique) e uma realizacao de Bernoulli. A funcao de perda Binary Cross-Entropy assume exatamente esse modelo.

**Conexao com secao 9:** Bernoulli e o tijolo; Binomial e a parede (n Bernoullis somados).

In [ ]:
# Distribuição Bernoulli
p = 0.3  # Probabilidade de sucesso

# Gerar 1000 amostras Bernoulli
samples = np.random.binomial(1, p, 1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PMF
ax1.bar([0, 1], [1-p, p], color=['orange', 'green'], alpha=0.7, width=0.4)
ax1.set_xticks([0, 1])
ax1.set_xlabel('Resultado')
ax1.set_ylabel('Probabilidade')
ax1.set_title(f'Bernoulli(p={p}): PMF')
ax1.set_ylim([0, 1])
for i, v in enumerate([1-p, p]):
    ax1.text(i, v + 0.05, f'{v:.2f}', ha='center', fontweight='bold')

# Sequência de amostras
ax2.plot(np.cumsum(samples) / np.arange(1, 1001), linewidth=2, label='Proporção acumulada')
ax2.axhline(p, color='red', linestyle='--', linewidth=2, label=f'p verdadeiro = {p}')
ax2.set_xlabel('Número de amostras')
ax2.set_ylabel('Proporção de sucessos')
ax2.set_title('Convergência para p pela Lei dos Grandes Números')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print(f'E[X] = {samples.mean():.4f} (teórico: {p})')
print(f'Var[X] = {samples.var():.4f} (teórico: {p*(1-p):.4f})')

**O que observar:** A esquerda, a PMF mostra dois picos em 0 e 1 com alturas 1-p e p. A direita, a media acumulada oscila inicialmente mas converge para p = 0.3 conforme n cresce.

**O que concluir:** A convergencia da media amostral para p e a Lei dos Grandes Numeros em acao (secao 15). A variancia p(1-p) e maxima quando p=0.5 -- e quando temos maxima incerteza sobre o resultado.

**Conexao com secao 14:** E[X] = p e Var[X] = p(1-p) sao os primeiros exemplos concretos de esperanca e variancia.

## 9. Distribuicao Binomial

### Intuicao: Repetindo o Experimento

Se Bernoulli e um lancamento de moeda, Binomial e lancar a moeda n vezes e contar caras.

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

O fator $\binom{n}{k}$ conta de quantas formas diferentes podemos ter k sucessos em n tentativas.

| Propriedade | Valor | Intuicao |
|-------------|-------|----------|
| E[X] | np | media = tentativas * taxa de sucesso |
| Var[X] | np(1-p) | espalhamento cresce com n |
| Moda | ≈ np | pico do sino |

**Por que em ML:** Numero de cliques em n impressoes de anuncio, numero de acertos em n predicoes, taxa de conversao.

**Conexao com secao 12:** Quando n e grande e p nao e extremo, Binomial se aproxima da Normal (Teorema Central do Limite).

In [ ]:
# Distribuição Binomial
n = 20  # número de tentativas
p = 0.3  # probabilidade de sucesso

# Gerar amostras
samples = np.random.binomial(n, p, 10_000)

# Teórico via scipy
x_vals = np.arange(0, n+1)
# # pmf_theory = stats.binom.pmf(x_vals, n, p)  # (commented - scipy.stats removed)

fig, ax = plt.subplots(figsize=(12, 6))

# Histograma empírico
ax.hist(samples, bins=np.arange(0, n+2)-0.5, density=True, alpha=0.7, 
        label='Empírico (10k amostras)', color='steelblue', edgecolor='black')

# PMF teórica
# ax.plot(x_vals, pmf_theory, 'o-', color='red', linewidth=2, markersize=8, label='Teórica')  # (commented - scipy.stats removed)

ax.set_xlabel('Número de sucessos (k)')
ax.set_ylabel('Probabilidade')
ax.set_title(f'Distribuição Binomial(n={n}, p={p})')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

mean_empirical = samples.mean()
mean_theory = n * p
var_empirical = samples.var()
var_theory = n * p * (1 - p)

print(f'E[X] empírico: {mean_empirical:.4f}, teórico: {mean_theory:.4f}')
print(f'Var[X] empírico: {var_empirical:.4f}, teórico: {var_theory:.4f}')
# print(f'P(X=6) = {pmf_theory[6]:.4f}')  # (commented - scipy.stats removed)
# print(f'P(X≤6) = {stats.binom.cdf(6, n, p):.4f}')

**O que observar:** O histograma empirico e a PMF teorica (pontos vermelhos) se sobrepoem quase perfeitamente. O formato e de sino com pico proximo a np = 6.

**O que concluir:** A formula binomial descreve com precisao a distribuicao de sucessos em tentativas independentes. A sobreposicao emperico-teorica confirma tanto a formula quanto a qualidade do gerador aleatorio.

**Conexao com secao 10:** Se n → infinito e p → 0 com np = lambda constante, Binomial converge para Poisson.

## 10. Distribuicao Poisson

### Intuicao: Contando Eventos Raros

Poisson modela o numero de eventos que ocorrem em um intervalo fixo, quando:
- Eventos sao independentes
- A taxa media e constante (lambda eventos por intervalo)
- Dois eventos nao ocorrem no mesmo instante

$$P(X = k) = \frac{e^{-\lambda} \lambda^k}{k!}$$

| Propriedade | Valor | Destaque |
|-------------|-------|----------|
| E[X] | lambda | media = taxa |
| Var[X] | lambda | **E[X] = Var[X]** (propriedade unica!) |

**Exemplos classicos:**
- Chamadas a um call center por minuto
- Falhas em um servidor por dia
- Erros de digitacao por pagina
- Acidentes em uma rodovia por mes

**Por que em ML:** Modelos de contagem (regressao Poisson) sao usados quando a variavel resposta e uma contagem nao-negativa.

In [ ]:
# Distribuição Poisson
lambda_param = 3.5  # eventos por unidade de tempo

# Gerar amostras
samples = np.random.poisson(lambda_param, 10_000)

# Teórico
x_vals = np.arange(0, 15)
# # pmf_theory = stats.poisson.pmf(x_vals, lambda_param)  # (commented - scipy.stats removed)

fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(samples, bins=np.arange(-0.5, 15.5, 1), density=True, alpha=0.7, 
        label='Empírico (10k amostras)', color='coral', edgecolor='black')
# ax.plot(x_vals, pmf_theory, 'o-', color='darkblue', linewidth=2, markersize=8, label='Teórica')  # (commented - scipy.stats removed)

ax.set_xlabel('Número de eventos')
ax.set_ylabel('Probabilidade')
ax.set_title(f'Distribuição Poisson(λ={lambda_param}) - Eventos raros')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

mean_empirical = samples.mean()
var_empirical = samples.var()

print(f'E[X] empírico: {mean_empirical:.4f}, teórico: {lambda_param:.4f}')
print(f'Var[X] empírico: {var_empirical:.4f}, teórico: {lambda_param:.4f}')
print(f'\nPropriedade importante: Em Poisson, E[X] = Var[X]')
# print(f'P(X=0) = {pmf_theory[0]:.4f}  (nenhum evento)')  # (commented - scipy.stats removed)
# print(f'P(X≥1) = {1 - pmf_theory[0]:.4f}  (pelo menos um evento)')  # (commented - scipy.stats removed)

**O que observar:** A distribuicao tem assimetria positiva (cauda para a direita). O pico esta proximo a lambda = 3.5. A propriedade E[X] = Var[X] e verificada empiricamente.

**O que concluir:** Poisson naturalmente prediz muitos zeros e poucos eventos grandes. Isso reflete a realidade de eventos raros. A igualdade E[X] = Var[X] e um teste rapido: se seus dados de contagem tem media muito diferente da variancia, Poisson pode nao ser adequado (considere Binomial Negativa).

**Conexao com secao 13:** Se Poisson conta eventos por intervalo, Exponencial mede o tempo entre eventos consecutivos. Sao o par discreto/continuo do mesmo fenomeno.

## 11. Variaveis Aleatorias Continuas

### Intuicao: A Grande Diferenca

Com variaveis discretas, cada valor tem probabilidade nao-zero: P(dado = 3) = 1/6.

Com variaveis continuas, a probabilidade de um valor exato e zero: P(altura = 1.7500...m) = 0. Isso porque ha infinitos valores possiveis.

A solucao e usar **densidade**: f(x) mede "quanta probabilidade por unidade de x".

| Conceito | Discreto | Continuo |
|----------|----------|----------|
| Funcao principal | PMF: P(X = k) | PDF: f(x) |
| Probabilidade de intervalo | Soma | Integral: P(a < X < b) = integral de f(x) |
| CDF | F(k) = soma P(X <= k) | F(x) = integral de f(t) de -inf a x |
| Normalizacao | soma P = 1 | integral f(x) = 1 |

**Por que em ML:** A maioria das features em ML (idade, peso, preco, pixels) sao continuas. Entender PDF/CDF e essencial para modelagem estatistica.

**Conexao com 0.5:** Aqui usamos integrais (notebook 0.5) pela primeira vez em contexto probabilistico.

In [ ]:
# Exemplo: Tempo de espera num caixa (Exponencial)
lambda_param = 0.5  # clientes por minuto

# Gerar amostras
samples = np.random.exponential(1/lambda_param, 5000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PDF
x = np.linspace(0, 10, 1000)
pdf = lambda_param * np.exp(-lambda_param * x)
ax1.hist(samples, bins=50, density=True, alpha=0.7, label='Empírico', color='steelblue', edgecolor='black')
ax1.plot(x, pdf, color='red', linewidth=2, label='PDF teórica')
ax1.set_xlabel('Tempo de espera (minutos)')
ax1.set_ylabel('Densidade de probabilidade')
ax1.set_title('PDF: Função Densidade de Probabilidade')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_xlim([0, 10])

# CDF
cdf = 1 - np.exp(-lambda_param * x)
ax2.plot(x, cdf, color='red', linewidth=2, label='CDF teórica')
ax2.set_xlabel('Tempo de espera (minutos)')
ax2.set_ylabel('F(x) = P(X ≤ x)')
ax2.set_title('CDF: Função Distribuição Acumulada')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_xlim([0, 10])

plt.tight_layout()
plt.show()

print(f'P(espera ≤ 2 min) = {1 - np.exp(-lambda_param * 2):.4f}')
print(f'P(espera > 3 min) = {np.exp(-lambda_param * 3):.4f}')

**O que observar:** A PDF comeca alta e decai exponencialmente. A CDF cresce de 0 ate 1 em formato de "S" alongado. A area sob a PDF ate qualquer ponto x coincide com o valor da CDF em x.

**O que concluir:** PDF e CDF sao representacoes equivalentes: PDF mostra a "concentracao local" de probabilidade, CDF mostra a "acumulacao total". Para calcular P(2 < X < 5), basta fazer F(5) - F(2).

**Conexao com secao 12:** A Normal e o caso mais importante de variavel continua, com PDF em forma de sino.

## 12. Distribuicao Normal (Gaussiana)

### Intuicao: A Distribuicao Ubiqua

Por que a Normal aparece em TUDO?

Porque o **Teorema Central do Limite** (TCL) diz: se voce soma muitas influencias independentes (mesmo nao-normais), o resultado tende a Normal.

- Altura de pessoas = genes + alimentacao + exercicio + ... → Normal
- Erros de medicao = muitas fontes pequenas de erro → Normal
- Residuos de regressao = muitos fatores nao-modelados → Normal

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

| Parametro | Significado | Efeito na curva |
|-----------|-------------|-----------------|
| mu | Media (centro) | Desloca horizontalmente |
| sigma | Desvio-padrao | Alarga/estreita a curva |

**Regra Empirica 68-95-99.7:**
- 68% dos dados em [mu - sigma, mu + sigma]
- 95% dos dados em [mu - 2*sigma, mu + 2*sigma]
- 99.7% dos dados em [mu - 3*sigma, mu + 3*sigma]

**Por que em ML:** Muitas funcoes de perda (MSE) e metodos (regressao linear) assumem erros normais. A Normal e a distribuicao "default" quando nao se sabe a distribuicao.

In [ ]:
# Distribuição Normal
mu, sigma = 100, 15  # IQ típico: média 100, desvio 15

# Gerar amostras
samples = np.random.normal(mu, sigma, 10_000)

fig, ax = plt.subplots(figsize=(12, 6))

# Histograma
ax.hist(samples, bins=50, density=True, alpha=0.7, label='Empírico', 
        color='steelblue', edgecolor='black')

# PDF teórica
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 1000)
# pdf = stats.norm.pdf(x, mu, sigma)
ax.plot(x, pdf, color='red', linewidth=3, label='PDF teórica')

# Adicionar regiões 68-95-99.7
for n_sigma, alpha, color in [(1, 0.3, 'green'), (2, 0.2, 'orange'), (3, 0.1, 'red')]:
    x_fill = x[(x >= mu - n_sigma*sigma) & (x <= mu + n_sigma*sigma)]
#     ax.fill_between(x_fill, stats.norm.pdf(x_fill, mu, sigma), alpha=alpha)
    
ax.axvline(mu, color='black', linestyle='--', linewidth=2, label=f'μ = {mu}')
ax.set_xlabel('QI')
ax.set_ylabel('Densidade de probabilidade')
ax.set_title(f'Distribuição Normal(μ={mu}, σ={sigma}) - Regra 68-95-99.7')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# print(f'P(μ - σ ≤ X ≤ μ + σ) = {stats.norm.cdf(mu + sigma, mu, sigma) - stats.norm.cdf(mu - sigma, mu, sigma):.4f}  ≈ 68%')
# print(f'P(μ - 2σ ≤ X ≤ μ + 2σ) = {stats.norm.cdf(mu + 2*sigma, mu, sigma) - stats.norm.cdf(mu - 2*sigma, mu, sigma):.4f}  ≈ 95%')
# print(f'P(μ - 3σ ≤ X ≤ μ + 3σ) = {stats.norm.cdf(mu + 3*sigma, mu, sigma) - stats.norm.cdf(mu - 3*sigma, mu, sigma):.4f}  ≈ 99.7%')
# print(f'\nP(X > 130) = {1 - stats.norm.cdf(130, mu, sigma):.4f}')
# print(f'P(X < 70) = {stats.norm.cdf(70, mu, sigma):.4f}')

**O que observar:** As regioes coloridas mostram as bandas 68-95-99.7. A maioria dos dados (68%) cabe em mais/menos 1 sigma. Valores alem de 3 sigma sao extremamente raros.

**O que concluir:** A regra 68-95-99.7 e uma ferramenta pratica para deteccao de outliers: qualquer valor alem de 3 sigma merece investigacao. Em ML, isso e usado em Z-score normalization e na definicao de valores anomalos.

**Conexao com secao 15:** A Lei dos Grandes Numeros diz que a media converge; o TCL diz que a distribuicao da media e Normal. Juntos, eles justificam praticamente toda a inferencia estatistica.

## 13. Distribuicao Exponencial

### Intuicao: Tempo ate o Proximo Evento

Se Poisson conta *quantos* eventos ocorrem por intervalo, Exponencial mede *quanto tempo* ate o proximo evento.

$$f(x) = \lambda e^{-\lambda x}, \quad x \geq 0$$

| Propriedade | Valor |
|-------------|-------|
| E[X] | 1/lambda |
| Var[X] | 1/lambda^2 |
| Propriedade especial | **Memoria zero** |

**Memoria zero:** P(X > s + t dado X > s) = P(X > t). O tempo de espera adicional nao depende de quanto voce ja esperou. E como se o processo "esquecesse" o passado.

**Por que em ML:** Modelagem de tempos de espera, analise de sobrevivencia, processos de filas. A propriedade de memoria zero simplifica muitos calculos.

In [ ]:
# Distribuição Exponencial
lambda_vals = [0.2, 0.5, 1.0, 2.0]  # diferentes taxas

fig, ax = plt.subplots(figsize=(12, 6))

for lam in lambda_vals:
    x = np.linspace(0, 10, 1000)
    pdf = lam * np.exp(-lam * x)
    ax.plot(x, pdf, linewidth=2.5, label=f'λ = {lam}')

ax.set_xlabel('Tempo')
ax.set_ylabel('f(x)')
ax.set_title('Distribuição Exponencial com diferentes taxas λ')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_xlim([0, 10])
ax.set_ylim([0, 2])

plt.tight_layout()
plt.show()

# Propriedade sem memória
lambda_param = 0.5
print(f'Propriedade da Memória Zero:')
print(f'P(X > 5) = {np.exp(-lambda_param * 5):.4f}')
print(f'P(X > 5 + 3 | X > 5) = P(X > 3) = {np.exp(-lambda_param * 3):.4f}')
print(f'A probabilidade de esperar 3 minutos a mais é independente')
print(f'do tempo já esperado!')
print(f'\nE[X] = {1/lambda_param:.4f}')
print(f'Var[X] = {1/lambda_param**2:.4f}')

**O que observar:** Lambda maior = decaimento mais rapido (eventos mais frequentes, esperas mais curtas). Todas as curvas comecam em lambda e decaem exponencialmente.

**O que concluir:** A propriedade de memoria zero e verificada numericamente: P(X > 8 dado X > 5) = P(X > 3), independente de quando comecamos a contar. Isso e unico da Exponencial entre as distribuicoes continuas.

**Conexao com secao 10:** Poisson (contagem) e Exponencial (tempo entre eventos) sao um par natural. Se chegadas seguem Poisson(lambda), os intervalos entre chegadas seguem Exponencial(lambda).

## 14. Esperanca e Variancia

### Intuicao: Resumindo uma Distribuicao em Dois Numeros

Uma distribuicao pode ter infinitos valores possiveis. Como resumi-la?

**Esperanca (media):** "centro de massa" da distribuicao.
$$E[X] = \sum_k k \cdot P(X=k) \quad \text{(discreta)}$$
$$E[X] = \int_{-\infty}^{\infty} x \cdot f(x)\, dx \quad \text{(continua)}$$

**Variancia:** "espalhamento" ao redor da media.
$$\text{Var}[X] = E[(X - E[X])^2] = E[X^2] - (E[X])^2$$

**Propriedades essenciais:**

| Propriedade | Formula | Quando usar |
|-------------|---------|-------------|
| Linearidade da esperanca | E[aX + b] = aE[X] + b | Normalizacao de features |
| Variancia com escala | Var[aX + b] = a^2 Var[X] | Propagacao de incerteza |
| Soma de esperancas | E[X + Y] = E[X] + E[Y] | Sempre (mesmo dependentes!) |
| Soma de variancias | Var[X + Y] = Var[X] + Var[Y] | Apenas se independentes |

**Por que em ML:** Funcoes de perda como MSE minimizam E[(Y - Yhat)^2]. Batch normalization usa E[X] e Var[X] dos mini-batches. A relacao vies-variancia (bias-variance tradeoff) e fundamental.

In [ ]:
# Demonstrar esperança e variância com simulação
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Bernoulli
ax = axes[0, 0]
p = 0.7
samples = np.random.binomial(1, p, 10_000)
ax.hist(samples, bins=2, alpha=0.7, edgecolor='black')
ax.axvline(np.mean(samples), color='red', linestyle='--', linewidth=2, label=f'E[X] = {np.mean(samples):.4f}')
ax.set_title(f'Bernoulli(p={p}): E[X]={p}, Var[X]={p*(1-p):.4f}')
ax.legend()
ax.grid(alpha=0.3)

# 2. Binomial
ax = axes[0, 1]
n, p = 20, 0.3
samples = np.random.binomial(n, p, 10_000)
ax.hist(samples, bins=20, alpha=0.7, edgecolor='black')
ax.axvline(np.mean(samples), color='red', linestyle='--', linewidth=2, label=f'E[X] = {np.mean(samples):.2f}')
ax.set_title(f'Binomial(n={n}, p={p}): E[X]={n*p}, Var[X]={n*p*(1-p):.2f}')
ax.legend()
ax.grid(alpha=0.3)

# 3. Poisson
ax = axes[0, 2]
lambda_param = 4
samples = np.random.poisson(lambda_param, 10_000)
ax.hist(samples, bins=15, alpha=0.7, edgecolor='black')
ax.axvline(np.mean(samples), color='red', linestyle='--', linewidth=2, label=f'E[X] = {np.mean(samples):.2f}')
ax.set_title(f'Poisson(λ={lambda_param}): E[X]=Var[X]={lambda_param}')
ax.legend()
ax.grid(alpha=0.3)

# 4. Uniforme
ax = axes[1, 0]
a, b = 0, 10
samples = np.random.uniform(a, b, 10_000)
ax.hist(samples, bins=50, alpha=0.7, edgecolor='black')
ax.axvline(np.mean(samples), color='red', linestyle='--', linewidth=2, label=f'E[X] = {np.mean(samples):.2f}')
theor_mean = (a + b) / 2
theor_var = (b - a)**2 / 12
ax.set_title(f'Uniforme[{a},{b}]: E[X]={theor_mean:.1f}, Var[X]={theor_var:.2f}')
ax.legend()
ax.grid(alpha=0.3)

# 5. Normal
ax = axes[1, 1]
mu, sigma = 100, 15
samples = np.random.normal(mu, sigma, 10_000)
ax.hist(samples, bins=50, alpha=0.7, edgecolor='black')
ax.axvline(np.mean(samples), color='red', linestyle='--', linewidth=2, label=f'E[X] = {np.mean(samples):.2f}')
ax.set_title(f'Normal(μ={mu}, σ={sigma}): E[X]={mu}, Var[X]={sigma**2}')
ax.legend()
ax.grid(alpha=0.3)

# 6. Exponencial
ax = axes[1, 2]
lambda_param = 0.5
samples = np.random.exponential(1/lambda_param, 10_000)
ax.hist(samples, bins=50, alpha=0.7, edgecolor='black')
ax.axvline(np.mean(samples), color='red', linestyle='--', linewidth=2, label=f'E[X] = {np.mean(samples):.2f}')
theor_mean = 1 / lambda_param
theor_var = 1 / lambda_param**2
ax.set_title(f'Exponencial(λ={lambda_param}): E[X]={theor_mean:.1f}, Var[X]={theor_var:.1f}')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**O que observar:** Cada distribuicao tem formato diferente, mas todas podem ser resumidas por E[X] (linha vermelha) e Var[X] (espalhamento). A Normal e simetrica; Exponencial e assimetrica; Poisson tem E = Var.

**O que concluir:** Esperanca e variancia sao universais -- aplicam-se a qualquer distribuicao. Mas cuidado: distribuicoes com mesma media e variancia podem ter formatos muito diferentes. Para uma descricao completa, precisamos de momentos superiores (assimetria, curtose) ou da distribuicao inteira.

**Tabela-resumo das distribuicoes:**

| Distribuicao | E[X] | Var[X] | Tipo |
|-------------|------|--------|------|
| Bernoulli(p) | p | p(1-p) | Discreta |
| Binomial(n,p) | np | np(1-p) | Discreta |
| Poisson(lambda) | lambda | lambda | Discreta |
| Uniforme[a,b] | (a+b)/2 | (b-a)^2/12 | Continua |
| Normal(mu,sigma) | mu | sigma^2 | Continua |
| Exponencial(lambda) | 1/lambda | 1/lambda^2 | Continua |

**Conexao com secao 15:** A LGN garante que a media amostral converge para E[X] quando n → infinito.

## 15. Lei dos Grandes Numeros (LGN)

### Intuicao: Mais Dados = Melhor Estimativa

A LGN formaliza algo que parece obvio: quanto mais dados coletamos, mais nossa media amostral se aproxima da media verdadeira.

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i \xrightarrow{n \to \infty} E[X]$$

**Analogia:** Se voce quer saber a proporcao de carros vermelhos na cidade, pode contar 10 carros (estimativa ruim) ou 10.000 carros (estimativa otima). A LGN garante que 10.000 e melhor.

**Versoes:**
- **LGN Fraca:** convergencia em probabilidade (a maioria das amostras converge)
- **LGN Forte:** convergencia quase certa (todas as amostras convergem, exceto conjunto de medida zero)

**Por que em ML:** A LGN justifica:
- Usar media dos dados para estimar parametros
- Mini-batch SGD (media do gradiente no batch converge para o gradiente verdadeiro)
- Validacao cruzada (media dos folds converge para o desempenho real)
- Qualquer estimativa baseada em amostragem

In [ ]:
# Demonstração da Lei dos Grandes Números
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Testar com 4 distribuições
distributions = [
    ('Bernoulli(p=0.3)', np.random.binomial, {'n': 1, 'p': 0.3}, 0.3),
    ('Uniforme[0,10]', np.random.uniform, {'low': 0, 'high': 10}, 5),
    ('Normal(μ=50, σ=5)', np.random.normal, {'loc': 50, 'scale': 5}, 50),
    ('Poisson(λ=3)', np.random.poisson, {'lam': 3}, 3)
]

for idx, (name, dist_func, params, true_mean) in enumerate(distributions):
    ax = axes[idx // 2, idx % 2]
    
    # Gerar amostras sequencialmente
    n_max = 5000
    samples = dist_func(**params, size=n_max)
    cumulative_mean = np.cumsum(samples) / np.arange(1, n_max + 1)
    
    ax.plot(cumulative_mean, linewidth=2, alpha=0.8, label='Média acumulada')
    ax.axhline(true_mean, color='red', linestyle='--', linewidth=2, label=f'E[X] = {true_mean}')
    ax.fill_between(np.arange(n_max), true_mean - 0.5, true_mean + 0.5, 
                     alpha=0.1, color='green', label='Intervalo ±0.5')
    
    ax.set_xlabel('Número de amostras (n)')
    ax.set_ylabel('Média acumulada')
    ax.set_title(f'LGN: {name}')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_xlim([0, n_max])

plt.tight_layout()
plt.show()

print('A Lei dos Grandes Números garante que:')
print('1. Mais dados → estimativas mais precisas')
print('2. Não há limite teórico (apenas convergência lenta)')
print('3. Funciona para qualquer distribuição!')
print('4. Justifica o uso de média amostral em ML!')

**O que observar:** As quatro distribuicoes (Bernoulli, Uniforme, Normal, Poisson) mostram o mesmo padrao: a media acumulada oscila muito no inicio mas converge para o valor teorico (linha vermelha) conforme n cresce.

**O que concluir:** A LGN funciona para QUALQUER distribuicao com esperanca finita. Nao importa se e discreta, continua, simetrica ou assimetrica. Isso e extraordinariamente poderoso: podemos estimar parametros desconhecidos simplesmente coletando dados suficientes.

**Conexao com 0.7:** No proximo notebook, veremos como quantificar a velocidade de convergencia (Desigualdade de Chebyshev) e como o Teorema Central do Limite descreve a distribuicao da media amostral.

## 16. Exercicios Finais

### Exercicio 1: Diagnostico Medico com Bayes

Um teste para detectar certa doenca rara tem:
- Sensibilidade: 99% (P(+ | doente) = 0.99)
- Especificidade: 98% (P(- | saudavel) = 0.98)
- Prevalencia: 0.1% (P(doente) = 0.001)

**Tarefas:**
1. Calcule P(doente | +) usando a regra de Bayes
2. Interprete o resultado: o que significa clinicamente?
3. Que percentual de resultados positivos sao falsos positivos?
4. Se a prevalencia subir para 5%, como muda P(doente | +)?

*Dica: Use a formula de Bayes com a lei da probabilidade total no denominador.*

In [ ]:
# EXERCICIO 1 - Sua pratica
# Calcule P(doente | +) usando Bayes

# Dados do problema
p_doente = None        # TAREFA DO ALUNO: prevalencia
p_pos_doente = None    # TAREFA DO ALUNO: sensibilidade
p_neg_saudavel = None  # TAREFA DO ALUNO: especificidade

# TAREFA DO ALUNO: Calcule P(+ | saudavel) = 1 - especificidade
p_pos_saudavel = None

# TAREFA DO ALUNO: Calcule P(+) pela lei da probabilidade total
# P(+) = P(+|doente)*P(doente) + P(+|saudavel)*P(saudavel)
p_positivo = None

# TAREFA DO ALUNO: Aplique Bayes
p_doente_dado_pos = None

# TAREFA DO ALUNO: Repita com prevalencia = 5%
# ...

print(f'P(doente | +) = {p_doente_dado_pos}')

In [ ]:
# Solução Exercício 1
p_disease = 0.001
p_pos_given_disease = 0.99
p_pos_given_healthy = 1 - 0.98  # = 0.02
p_healthy = 1 - p_disease

p_positive = (p_pos_given_disease * p_disease + 
              p_pos_given_healthy * p_healthy)
p_disease_given_positive = (p_pos_given_disease * p_disease) / p_positive

print('Exercício 1: Diagnóstico Médico')
print(f'P(doente | +) = {p_disease_given_positive:.6f}  ({p_disease_given_positive*100:.2f}%)')
print(f'\nInterpretação: Mesmo com sensibilidade/especificidade altas,')
print(f'um resultado positivo em população com baixa prevalência')
print(f'ainda indica apenas ~5% de chance de ter a doença!')
print(f'Motivo: falsos positivos são frequentes quando a doença é rara.')
print(f'\nRecomendação clínica: Fazer segundo teste para confirmar!')

# Visualizar
fig, ax = plt.subplots(figsize=(10, 6))
labels = ['Antes do teste\n(Prior)', 'Depois do teste +\n(Posterior)']
values = [p_disease, p_disease_given_positive]
colors = ['lightcoral', 'darkred']
ax.bar(labels, values, color=colors, alpha=0.7, width=0.5)
ax.set_ylabel('Probabilidade de doença')
ax.set_title('Atualização Bayesiana com Teste Médico')
ax.set_ylim([0, max(values) * 1.3])
for i, v in enumerate(values):
    ax.text(i, v + 0.005, f'{v*100:.3f}%', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Exercicio 2: Simulacao Binomial e Convergencia

Gere 5000 amostras de Binomial(n=50, p=0.4).

**Tarefas:**
1. Calcule media e variancia amostrais
2. Compare com os valores teoricos E[X] = np e Var[X] = np(1-p)
3. Crie um histograma com a PMF teorica sobreposta
4. Bonus: mostre que para n grande, Binomial se aproxima da Normal

*Dica: Use stats.binom.pmf para a PMF teorica e stats.norm.pdf para a Normal aproximada.*

In [ ]:
# EXERCICIO 2 - Sua pratica
# Simulacao Binomial

n, p = 50, 0.4

# TAREFA DO ALUNO: Gere 5000 amostras
samples = None

# TAREFA DO ALUNO: Calcule media e variancia amostrais
mean_emp = None
var_emp = None

# TAREFA DO ALUNO: Compare com valores teoricos
mean_teo = None  # np
var_teo = None   # np(1-p)

# TAREFA DO ALUNO: Crie histograma + PMF teorica
# fig, ax = plt.subplots(...)
# ax.hist(samples, ...)
# ax.plot(x_vals, stats.binom.pmf(...), ...)

print(f'Media empirica: {mean_emp}, teorica: {mean_teo}')
print(f'Variancia empirica: {var_emp}, teorica: {var_teo}')

In [ ]:
# Solução Exercício 2
n, p = 50, 0.4
samples = np.random.binomial(n, p, 5000)

mean_empirical = samples.mean()
mean_theory = n * p
var_empirical = samples.var()
var_theory = n * p * (1 - p)

print('Exercício 2: Simulação Binomial')
print(f'\nResultados empíricos (5000 amostras):')
print(f'Média: {mean_empirical:.4f}')
print(f'Variância: {var_empirical:.4f}')
print(f'\nValores teóricos:')
print(f'E[X] = np = {n}*{p} = {mean_theory:.4f}')
print(f'Var[X] = np(1-p) = {n}*{p}*{1-p} = {var_theory:.4f}')
print(f'\nErros:')
print(f'Erro em média: {abs(mean_empirical - mean_theory):.4f}')
print(f'Erro em variância: {abs(var_empirical - var_theory):.4f}')

# Visualização
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(samples, bins=30, density=True, alpha=0.7, label='Empírico', edgecolor='black')
x_vals = np.arange(0, n+1)
# # pmf_theory = stats.binom.pmf(x_vals, n, p)  # (commented - scipy.stats removed)
# ax.plot(x_vals, pmf_theory, 'ro-', linewidth=2, markersize=6, label='Teórica')  # (commented - scipy.stats removed)
ax.set_xlabel('Número de sucessos')
ax.set_ylabel('Probabilidade')
ax.set_title(f'Exercício 2: Binomial(n={n}, p={p}) - Comparação empírico vs teórico')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Exercicio 3: Estimacao de Parametros por Amostragem

Voce recebe amostras de uma distribuicao Normal desconhecida (nao sabe mu nem sigma).

**Tarefas:**
1. Para tamanhos de amostra n = 10, 50, 100, 500, estime mu e sigma
2. Plote histograma com PDF estimada sobreposta para cada n
3. Observe como a qualidade da estimativa melhora com n
4. Bonus: calcule o erro absoluto medio das estimativas para cada n

*Dica: Use np.mean e np.std(ddof=1) para estimadores nao-viesados.*

In [ ]:
# EXERCICIO 3 - Sua pratica
# Estimacao de parametros

# Geracao (nao altere - simula dados "desconhecidos")
np.random.seed(123)
mu_true = 50
sigma_true = 8

sample_sizes = [10, 50, 100, 500]

# TAREFA DO ALUNO: Para cada n em sample_sizes:
#   1. Gere amostra: np.random.normal(mu_true, sigma_true, n)
#   2. Estime mu_hat = np.mean(amostra)
#   3. Estime sigma_hat = np.std(amostra, ddof=1)
#   4. Plote histograma + PDF estimada
#   5. Calcule erro = |mu_hat - mu_true|

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# for idx, n in enumerate(sample_sizes):
#     ...

In [ ]:
# Solução Exercício 3
# # from scipy.stats import kstest

# Simular: amostra de Normal desconhecida
mu_true = 50
sigma_true = 8
sample_sizes = [10, 50, 100, 500, 1000, 5000]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, n in enumerate(sample_sizes):
    if idx >= 4:
        break
    ax = axes[idx // 2, idx % 2]
    
    # Gerar amostra
    sample = np.random.normal(mu_true, sigma_true, n)
    
    # Estimadores
    mu_hat = np.mean(sample)
    sigma_hat = np.std(sample, ddof=1)  # ddof=1 para estimador não-enviesado
    
    # Plotar
    ax.hist(sample, bins=20, density=True, alpha=0.7, edgecolor='black', label='Amostra')
    x = np.linspace(mu_true - 4*sigma_true, mu_true + 4*sigma_true, 1000)
#     ax.plot(x, stats.norm.pdf(x, mu_hat, sigma_hat), 'r-', linewidth=2, label='PDF estimada')
    ax.axvline(mu_hat, color='green', linestyle='--', linewidth=2, alpha=0.7)
    
    ax.set_title(f'n = {n}\nμ̂ = {mu_hat:.2f} (verdadeiro: {mu_true})\nσ̂ = {sigma_hat:.2f} (verdadeiro: {sigma_true})')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('Exercício 3: Estimação de Parâmetros')
print(f'\nVerdadeiros parâmetros: μ = {mu_true}, σ = {sigma_true}')
print(f'\nA medida que aumentamos n, nossas estimativas convergem')
print(f'aos valores verdadeiros (Lei dos Grandes Números!).')
print(f'\nEstá é a base da inferência estatística em ML:')  
print(f'dados + modelos → estimativas de parâmetros')
print(f'mais dados → estimativas mais precisas.')

## 17. Erros Comuns

### Erro 1: Confundir P(A|B) com P(B|A)
**Sintoma:** "O teste deu positivo, logo tenho a doenca com 95% de certeza"
**Causa:** P(+ | doente) = 0.95 NAO implica P(doente | +) = 0.95
**Correcao:** Sempre aplique Bayes. O prior (prevalencia) importa enormemente

### Erro 2: Assumir independencia sem verificar
**Sintoma:** Multiplicar probabilidades de features diretamente
**Causa:** Features correlacionadas violam P(A,B) = P(A)*P(B)
**Correcao:** Naive Bayes funciona apesar disso para classificacao, mas para estimativas pontuais de probabilidade, use modelos que capturam dependencias

### Erro 3: Confundir variancia com probabilidade
**Sintoma:** "Variancia = 25, entao 25% dos dados estao aqui"
**Causa:** Variancia tem unidades ao quadrado e nao e uma probabilidade
**Correcao:** Use desvio-padrao (sigma = raiz de Var) e a regra 68-95-99.7

### Erro 4: Usar distribuicao errada
**Sintoma:** Usar Binomial para contagem de eventos por tempo
**Causa:** Binomial requer n fixo; Poisson modela contagens sem limite
**Correcao:** Pergunte: "Tenho n fixo?" Se sim → Binomial. Se nao → Poisson

### Erro 5: Esquecer normalizacao
**Sintoma:** Soma de P(X=k) nao resulta em 1
**Causa:** Erro de implementacao ou distribuicao mal definida
**Correcao:** Sempre verifique que soma P = 1 (discreta) ou integral f = 1 (continua)

### Erro 6: Confundir PDF com probabilidade
**Sintoma:** "f(x) = 1.5, entao a probabilidade e 150%?"
**Causa:** PDF e DENSIDADE, nao probabilidade. Pode ser > 1
**Correcao:** Probabilidade e a AREA sob a curva: P(a < X < b) = integral de a a b de f(x)dx

### Erro 7: Ignorar o Prior em Bayes
**Sintoma:** Focar apenas na likelihood e ignorar a prevalencia
**Causa:** Viés de confirmacao -- queremos que a evidencia confirme nossa hipotese
**Correcao:** Sempre inclua P(A) no calculo. Para doencas raras, o prior domina

## 18. Resumo e Conexoes

### Diagrama de Dependencias Conceituais

```
Axiomas de Kolmogorov (secao 2)
    |
    v
Regras basicas: Uniao, Complemento (secao 3)
    |           \
    v            v
Condicional     Independencia (secao 6)
(secao 4)           |
    |               v
    v           Naive Bayes
Regra de Bayes      |
(secao 5)           |
    |               |
    +-------+-------+
            |
            v
    Variaveis Aleatorias
    /                   \
   v                     v
Discretas (secao 7)   Continuas (secao 11)
   |                     |
   v                     v
Bernoulli → Binomial   Exponencial  Normal
(sec 8)    (sec 9)     (sec 13)    (sec 12)
   |          |            |           |
   v          v            v           v
        Poisson         Par Poisson-Exponencial
        (sec 10)
            \               /
             v             v
       Esperanca e Variancia (secao 14)
                |
                v
    Lei dos Grandes Numeros (secao 15)
```

### Tabela de Conexoes com Outros Notebooks

| Conceito deste notebook | Onde sera usado | Como |
|------------------------|-----------------|------|
| P(Y dado X) condicional | 0.7, 1.1 | Base de toda classificacao |
| Regra de Bayes | 0.7 (Inferencia Bayesiana) | Atualizacao de parametros |
| Normal | 0.7 (Multivariada), 1.1 (Testes) | Distribuicao padrao de erros |
| E[X] e Var[X] | 0.7, 0.8 | Funcoes de perda e regularizacao |
| LGN | 0.8 (SGD), 1.1 (Amostragem) | Justifica uso de mini-batches |
| Binomial/Poisson | 1.1 (Testes discretos) | Modelos de contagem |
| Bernoulli | 0.8 (Binary Cross-Entropy) | Base de classificacao binaria |

### Checklist de Verificacao

Antes de prosseguir para 0.7, verifique que consegue:

- [ ] Enunciar os 3 axiomas de Kolmogorov e explicar por que sao suficientes
- [ ] Aplicar regra da uniao e complemento em problemas praticos
- [ ] Calcular P(A|B) e explicar a diferenca de P(B|A)
- [ ] Aplicar Bayes e interpretar o resultado (prior vs posterior)
- [ ] Explicar por que Naive Bayes funciona apesar da suposicao "ingenue"
- [ ] Distinguir PMF de PDF e saber quando usar cada uma
- [ ] Para cada distribuicao (Bernoulli, Binomial, Poisson, Normal, Exponencial): saber parametros, E[X], Var[X], e quando usar
- [ ] Calcular E[X] e Var[X] e usar propriedades de linearidade
- [ ] Explicar a LGN e por que ela justifica uso de dados em ML
- [ ] Resolver problemas que combinam multiplos conceitos (como Exercicio 1)

### Proximos Passos

- **0.7 (Probabilidade Avancada):** Distribuicoes multivariadas, MLE, MAP, Inferencia Bayesiana
- **0.8 (Otimizacao para ML):** Como usar gradientes para treinar modelos
- **1.1 (Estatistica Inferencial):** Testes de hipotese, intervalos de confianca

In [ ]:
# SOLUCAO 1: Probabilidade Condicional
print("=== SOLUCAO: P(A|B) ===")
P_A = 0.3  # P(Chover)
P_B = 0.4  # P(Sair)
P_A_and_B = 0.15  # P(Chover E Sair)

P_A_given_B = P_A_and_B / P_B
print(f"P(Chover | Sair) = {P_A_given_B:.4f}")
print(f"\nInterpretação: Se a pessoa sai, a probabilidade de chover é {P_A_given_B:.1%}")

In [ ]:
# SOLUCAO 2: Teorema de Bayes
print("\n=== SOLUCAO: Teorema de Bayes ===")
P_test_pos_disease = 0.95  # sensibilidade
P_test_neg_no_disease = 0.90  # especificidade
P_disease = 0.01  # prevalência

P_test_pos = P_test_pos_disease * P_disease + (1 - P_test_neg_no_disease) * (1 - P_disease)
P_disease_given_test_pos = (P_test_pos_disease * P_disease) / P_test_pos

print(f"P(Doença | Teste +) = {P_disease_given_test_pos:.4f}")
print(f"\nInterpretação: Apesar do teste positivo, há apenas {P_disease_given_test_pos:.1%} de chance de ter a doença")
print("Razão: A doença é rara (1% prevalência)")

In [ ]:
# SOLUCAO 3: Distribuição Binomial
print("\n=== SOLUCAO: Distribuição Binomial ===")
# Qual é a probabilidade de obter exatamente 5 sucessos em 10 tentativas com p=0.6?
from math import comb
n = 10
k = 5
p = 0.6

P_X_equals_5 = comb(n, k) * (p**k) * ((1-p)**(n-k))
print(f"P(X = 5) = C(10,5) * 0.6^5 * 0.4^5 = {P_X_equals_5:.6f}")

# Calcular também com numpy
successes = np.arange(0, n+1)
probabilities = np.array([comb(n, int(k_val)) * (p**k_val) * ((1-p)**(n-k_val)) 
                          for k_val in successes])

print(f"\nDistribuição completa:")
for k_val, prob in zip(successes, probabilities):
    print(f"  P(X = {k_val:2d}) = {prob:.6f}")

print(f"\nE(X) = np = {n*p}")
print(f"Var(X) = np(1-p) = {n*p*(1-p)}")

## Resumo: Hierarquia de Conceitos

### Fundamentos
- **Espaço Amostral**: Conjunto de todos os resultados possíveis
- **Eventos**: Subconjuntos do espaço amostral
- **Probabilidade**: Medida numérica entre 0 e 1

### Hierarquia de Probabilidades

```
Axiomas de Kolmogorov (Base)
    ↓
Probabilidade Clássica (casos favoráveis/total)
    ↓
Probabilidade Condicional P(A|B) = P(A∩B)/P(B)
    ↓
Independência: P(A∩B) = P(A)·P(B)
    ↓
Teorema de Bayes: P(A|B) = P(B|A)·P(A)/P(B)
    ↓
Distribuições de Probabilidade
    ├─ Discretas: Binomial, Poisson, Geométrica
    └─ Contínuas: Normal, Exponencial, Uniforme
```

### Propriedades Fundamentais
1. **Regra da Multiplicação**: P(A∩B) = P(A)·P(B|A)
2. **Regra da Adição**: P(A∪B) = P(A) + P(B) - P(A∩B)
3. **Lei da Probabilidade Total**: P(B) = Σ P(B|A_i)·P(A_i)
4. **Valor Esperado**: E(X) = Σ x·P(X=x)
5. **Variância**: Var(X) = E(X²) - [E(X)]²